In [7]:
import sys
from pathlib import Path
from src.data_loader import load_raw_data

df = load_raw_data()

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 34,967
Columns: 61


In [8]:
anomalies = df.loc[
    df["bowlruns"].eq(-3)
].copy()

print(f"Records with bowlruns = -3: {len(anomalies)}")

Records with bowlruns = -3: 6


In [9]:
columns_to_inspect = [
    "p_match",
    "year",
    "match_date",
    "inns",
    "over_num",
    "ball",
    "ball_id",
    "team_bat",
    "team_bowl",
    "bat",
    "bowl",
    "score",
    "batruns",
    "wide",
    "noball",
    "byes",
    "legbyes",
    "bowlruns",
    "outcome",
    "out",
    "dismissal",
]

available_columns = [
    column
    for column in columns_to_inspect
    if column in anomalies.columns
]

display(
    anomalies[available_columns]
    .sort_values(["p_match", "inns", "ball_id"])
    .reset_index(drop=True)
)

,p_match,year,match_date,inns,over_num,ball,ball_id,team_bat,team_bowl,bat,...,score,batruns,wide,noball,byes,legbyes,bowlruns,outcome,out,dismissal
0,1359495,2023,4/15/2023,1,11,5,10.05,Lucknow Super Giants,Punjab Kings,Krunal Pandya,...,1,0,0,1,0,4,-3,leg bye,False,NaN
1,1359528,2023,5/9/2023,2,13,3,12.03,Mumbai Indians,Royal Challengers Bengaluru,Nehal Wadhera,...,1,0,0,1,0,4,-3,leg bye,False,NaN
2,1359537,2023,5/16/2023,2,19,4,18.04,Mumbai Indians,Lucknow Super Giants,Tim David,...,1,0,0,1,4,0,-3,bye,False,NaN
3,1422134,2024,4/3/2024,2,15,3,14.03,Delhi Capitals,Kolkata Knight Riders,Tristan Stubbs,...,1,0,0,1,4,0,-3,bye,False,NaN
4,1426274,2024,4/21/2024,1,17,3,16.03,Kolkata Knight Riders,Royal Challengers Bengaluru,Andre Russell,...,1,0,0,1,4,0,-3,bye,False,NaN
5,1426298,2024,5/11/2024,2,3,2,2.02,Mumbai Indians,Kolkata Knight Riders,Rohit Sharma,...,1,0,0,1,4,0,-3,bye,False,NaN


In [10]:
df_check = df.copy()

df_check["candidate_bowlruns"] = (
    df_check["batruns"]
    + df_check["wide"]
    + df_check["noball"]
)

df_check["bowlruns_difference"] = (
    df_check["bowlruns"]
    - df_check["candidate_bowlruns"]
)

print("Difference distribution:")
display(
    df_check["bowlruns_difference"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nRecords where bowlruns differs from the candidate:")
mismatches = df_check.loc[
    df_check["bowlruns_difference"].ne(0)
]

print(f"Mismatch count: {len(mismatches):,}")

display(
    mismatches[
        [
            "p_match",
            "year",
            "match_date",
            "inns",
            "over_num",
            "ball",
            "ball_id",
            "score",
            "batruns",
            "wide",
            "noball",
            "byes",
            "legbyes",
            "bowlruns",
            "candidate_bowlruns",
            "outcome",
        ]
    ]
    .head(20)
)

Difference distribution:


bowlruns_difference
-4        6
-1        8
 0    34953
Name: count, dtype: int64


Records where bowlruns differs from the candidate:
Mismatch count: 14


,p_match,year,match_date,inns,over_num,ball,ball_id,score,batruns,wide,noball,byes,legbyes,bowlruns,candidate_bowlruns,outcome
1452,1359480,2023,4/3/2023,2,20,2,19.02,1,0,0,1,0,1,0,1,leg bye
1813,1359482,2023,4/5/2023,1,19,4,18.04,1,0,0,1,1,0,0,1,bye
4565,1359493,2023,4/14/2023,2,10,4,9.04,1,0,0,1,0,1,0,1,leg bye
4951,1359495,2023,4/15/2023,1,11,5,10.05,1,0,0,1,0,4,-3,1,leg bye
9151,1359512,2023,4/28/2023,1,16,5,15.05,1,0,0,1,0,1,0,1,leg bye
13024,1359528,2023,5/9/2023,2,13,3,12.03,1,0,0,1,0,4,-3,1,leg bye
15180,1359537,2023,5/16/2023,2,19,4,18.04,1,0,0,1,4,0,-3,1,bye
17044,1370350,2023,5/23/2023,1,20,4,19.04,1,0,0,1,1,0,0,1,bye
20870,1422131,2024,3/31/2024,1,6,2,5.02,1,0,0,1,0,1,0,1,leg bye
21786,1422134,2024,4/3/2024,2,15,3,14.03,1,0,0,1,4,0,-3,1,bye


In [11]:
# Identify records where the source and candidate bowler runs differ.
mismatches = df_check.loc[
    df_check["bowlruns_difference"].ne(0)
].copy()

# Identify records where KKR was bowling.
kkr_bowling_mismatches = mismatches.loc[
    mismatches["team_bowl"].eq("Kolkata Knight Riders")
].copy()

# Identify records where KKR was batting.
kkr_batting_mismatches = mismatches.loc[
    mismatches["team_bat"].eq("Kolkata Knight Riders")
].copy()

print(f"All mismatches: {len(mismatches)}")
print(f"KKR bowling mismatches: {len(kkr_bowling_mismatches)}")
print(f"KKR batting mismatches: {len(kkr_batting_mismatches)}")

print(
    "\nDifference in bowler runs across KKR bowling mismatches:",
    kkr_bowling_mismatches["candidate_bowlruns"].sum()
    - kkr_bowling_mismatches["bowlruns"].sum(),
)

print("\nKKR bowling records requiring investigation:")
display(
    kkr_bowling_mismatches[
        [
            "p_match",
            "year",
            "match_date",
            "inns",
            "over_num",
            "ball",
            "bat",
            "bowl",
            "score",
            "batruns",
            "noball",
            "byes",
            "legbyes",
            "bowlruns",
            "candidate_bowlruns",
        ]
    ]
)

All mismatches: 14
KKR bowling mismatches: 2
KKR batting mismatches: 3

Difference in bowler runs across KKR bowling mismatches: 8

KKR bowling records requiring investigation:


,p_match,year,match_date,inns,over_num,ball,bat,bowl,score,batruns,noball,byes,legbyes,bowlruns,candidate_bowlruns
21786,1422134,2024,4/3/2024,2,15,3,Tristan Stubbs,Varun Chakravarthy,1,0,1,4,0,-3,1
32261,1426298,2024,5/11/2024,2,3,2,Rohit Sharma,Vaibhav Arora,1,0,1,4,0,-3,1


In [12]:
# All deliveries bowled by KKR in the dataset.
kkr_bowling = df_check.loc[
    df_check["team_bowl"].eq("Kolkata Knight Riders")
].copy()

# Legal deliveries: exclude wides and no-balls.
kkr_bowling["legal_delivery"] = (
    kkr_bowling["wide"].eq(0)
    & kkr_bowling["noball"].eq(0)
)

# Aggregate source and candidate bowling figures by bowler.
bowler_comparison = (
    kkr_bowling.groupby("bowl")
    .agg(
        source_runs=("bowlruns", "sum"),
        candidate_runs=("candidate_bowlruns", "sum"),
        legal_balls=("legal_delivery", "sum"),
        deliveries=("ball_id", "count"),
    )
    .reset_index()
)

bowler_comparison["source_economy"] = (
    bowler_comparison["source_runs"]
    / bowler_comparison["legal_balls"]
    * 6
)

bowler_comparison["candidate_economy"] = (
    bowler_comparison["candidate_runs"]
    / bowler_comparison["legal_balls"]
    * 6
)

bowler_comparison["runs_difference"] = (
    bowler_comparison["candidate_runs"]
    - bowler_comparison["source_runs"]
)

bowler_comparison["economy_difference"] = (
    bowler_comparison["candidate_economy"]
    - bowler_comparison["source_economy"]
)

display(
    bowler_comparison.loc[
        bowler_comparison["bowl"].isin(
            ["Varun Chakravarthy", "Vaibhav Arora"]
        ),
        [
            "bowl",
            "source_runs",
            "candidate_runs",
            "legal_balls",
            "source_economy",
            "candidate_economy",
            "runs_difference",
            "economy_difference",
        ],
    ].round(3)
)

,bowl,source_runs,candidate_runs,legal_balls,source_economy,candidate_economy,runs_difference,economy_difference
15,Vaibhav Arora,418,422,276,9.087,9.174,4,0.087
16,Varun Chakravarthy,827,831,616,8.055,8.094,4,0.039


In [13]:
import pandas as pd

# Identify legal deliveries.
df["legal_delivery"] = (
    df["wide"].eq(0)
    & df["noball"].eq(0)
)

print(f"Total delivery records: {len(df):,}")
print(f"Legal deliveries: {df['legal_delivery'].sum():,}")
print(f"Illegal deliveries: {(~df['legal_delivery']).sum():,}")

print("\nLegal delivery versus ballfaced:")
display(
    pd.crosstab(
        df["legal_delivery"],
        df["ballfaced"],
        margins=True,
    )
)

print("\nDeliveries with both wide and no-ball recorded:")
display(
    df.loc[
        df["wide"].gt(0) & df["noball"].gt(0),
        ["p_match", "inns", "over_num", "ball", "wide", "noball"],
    ]
)

Total delivery records: 34,967
Legal deliveries: 33,437
Illegal deliveries: 1,530

Legal delivery versus ballfaced:


ballfaced,0,1,All
legal_delivery,,,
False,1355,175,1530
True,0,33437,33437
All,1355,33612,34967



Deliveries with both wide and no-ball recorded:


,p_match,inns,over_num,ball,wide,noball


In [14]:
# Batting dot-ball definition.
df["batting_dot"] = (
    df["ballfaced"].eq(1)
    & df["batruns"].eq(0)
)

# Bowling dot-ball definition.
df["bowling_dot"] = (
    df["legal_delivery"]
    & df["score"].eq(0)
)

print("Batting dot-ball records:", df["batting_dot"].sum())
print("Bowling dot-ball records:", df["bowling_dot"].sum())

print("\nBowling dot-ball classification:")
display(
    pd.crosstab(
        df["legal_delivery"],
        df["score"].eq(0),
        rownames=["Legal delivery"],
        colnames=["Team score is zero"],
    )
)

print("\nLegal deliveries where the batter scores zero but the team scores runs:")
display(
    df.loc[
        df["legal_delivery"]
        & df["batruns"].eq(0)
        & df["score"].gt(0),
        [
            "p_match",
            "inns",
            "over_num",
            "ball",
            "batruns",
            "byes",
            "legbyes",
            "score",
            "outcome",
        ],
    ].head(20)
)

Batting dot-ball records: 11808
Bowling dot-ball records: 11186

Bowling dot-ball classification:


Team score is zero,False,True
Legal delivery,,
False,1530,0
True,22251,11186



Legal deliveries where the batter scores zero but the team scores runs:


,p_match,inns,over_num,ball,batruns,byes,legbyes,score,outcome
1,1359475,1,1,2,0,0,1,1,leg bye
88,1359475,1,15,3,0,0,4,4,leg bye
101,1359475,1,17,4,0,1,0,1,bye
117,1359475,1,20,2,0,0,1,1,leg bye
130,1359475,2,2,3,0,0,1,1,leg bye
146,1359475,2,7,4,0,0,1,1,leg bye
237,1359475,2,19,2,0,0,4,4,leg bye
303,1359476,1,10,3,0,0,1,1,leg bye
406,1359476,2,7,1,0,0,1,1,leg bye
417,1359476,2,8,6,0,1,0,1,bye


In [15]:
wickets = df.loc[df["out"].eq(True)].copy()

print(f"Total wicket-event records: {len(wickets):,}")

print("\nDismissal types:")
display(
    wickets["dismissal"]
    .value_counts(dropna=False)
    .rename_axis("dismissal")
    .reset_index(name="count")
)

print("\nDismissal type versus bat_out:")
display(
    pd.crosstab(
        wickets["dismissal"],
        wickets["bat_out"],
        margins=True,
        dropna=False,
    )
)

print("\nWicket records with missing dismissal type:")
display(
    wickets.loc[
        wickets["dismissal"].isna(),
        [
            "p_match",
            "year",
            "match_date",
            "inns",
            "over_num",
            "ball",
            "bat",
            "bowl",
            "out",
            "bat_out",
            "dismissal",
            "p_out",
        ],
    ]
)

Total wicket-event records: 1,799

Dismissal types:


,dismissal,count
0,caught,1270
1,bowled,268
2,leg before wicket,115
3,run out,107
4,stumped,33
5,retired not out (hurt),2
6,retired out,2
7,hit wicket,1
8,obstructing the field,1



Dismissal type versus bat_out:


bat_out,False,True,All
dismissal,,,
bowled,0,268,268
caught,0,1270,1270
hit wicket,0,1,1
leg before wicket,0,115,115
obstructing the field,0,1,1
retired not out (hurt),2,0,2
retired out,1,1,2
run out,49,58,107
stumped,0,33,33



Wicket records with missing dismissal type:


,p_match,year,match_date,inns,over_num,ball,bat,bowl,out,bat_out,dismissal,p_out


In [16]:
BOWLER_CREDITED_DISMISSALS = {
    "caught",
    "bowled",
    "leg before wicket",
    "stumped",
    "hit wicket",
}

df["bowler_credited_wicket"] = (
    df["out"].eq(True)
    & df["dismissal"].isin(BOWLER_CREDITED_DISMISSALS)
)

print("Records marked out:", df["out"].eq(True).sum())

print(
    "Candidate bowler-credited wickets:",
    df["bowler_credited_wicket"].sum(),
)

print("\nDismissal classification:")

display(
    df.loc[df["out"].eq(True)]
    .groupby("dismissal", dropna=False)
    .agg(
        records=("out", "size"),
        bowler_credited=("bowler_credited_wicket", "sum"),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

print("\nRecords marked out but not credited to the bowler:")

display(
    df.loc[
        df["out"].eq(True)
        & ~df["bowler_credited_wicket"],
        [
            "p_match",
            "year",
            "match_date",
            "inns",
            "over_num",
            "ball",
            "bat",
            "bowl",
            "dismissal",
            "out",
            "bat_out",
            "p_out",
        ],
    ]
)

Records marked out: 1799
Candidate bowler-credited wickets: 1687

Dismissal classification:


,dismissal,records,bowler_credited
1,caught,1270,1270
0,bowled,268,268
3,leg before wicket,115,115
7,run out,107,0
8,stumped,33,33
5,retired not out (hurt),2,0
6,retired out,2,0
2,hit wicket,1,1
4,obstructing the field,1,0



Records marked out but not credited to the bowler:


,p_match,year,match_date,inns,over_num,ball,bat,bowl,dismissal,out,bat_out,p_out
1760,1359482,2023,4/5/2023,1,11,1,Shikhar Dhawan,Ravichandran Ashwin,retired not out (hurt),True,False,342619
1945,1359482,2023,4/5/2023,2,20,3,Shimron Hetmyer,Sam Curran,run out,True,True,670025
2305,1359484,2023,4/7/2023,1,20,1,Abdul Samad,Jaydev Unadkat,run out,True,False,1246528
3747,1359490,2023,4/11/2023,1,19,4,Kuldeep Yadav,Jason Behrendorff,run out,True,True,559235
3799,1359490,2023,4/11/2023,2,8,3,Ishan Kishan,Lalit Yadav,run out,True,True,720471
...,...,...,...,...,...,...,...,...,...,...,...,...
33791,1426306,2024,5/18/2024,2,13,6,Rachin Ravindra,Glenn Maxwell,run out,True,True,959767
33935,1426307,2024,5/19/2024,1,16,7,Shashank Singh,Nitish Kumar Reddy,run out,True,True,377534
34169,1426309,2024,5/21/2024,1,14,2,Abdul Samad,Sunil Narine,run out,True,False,446763
34493,1426310,2024,5/22/2024,2,14,1,Riyan Parag,Cameron Green,run out,True,False,1175488


In [17]:
candidate_wickets = df.loc[df["bowler_credited_wicket"]].copy()

print("Candidate bowler-credited wickets:", len(candidate_wickets))

print(
    "Candidate wickets with missing bowler:",
    candidate_wickets["bowl"].isna().sum()
)

print(
    "Candidate wickets with blank bowler:",
    candidate_wickets["bowl"]
    .astype("string")
    .str.strip()
    .eq("")
    .sum()
)

print("\nCandidate wickets by dismissal type:")

display(
    candidate_wickets
    .groupby("dismissal")
    .size()
    .sort_values(ascending=False)
    .rename("count")
    .reset_index()
)

Candidate bowler-credited wickets: 1687
Candidate wickets with missing bowler: 0
Candidate wickets with blank bowler: 0

Candidate wickets by dismissal type:


,dismissal,count
0,caught,1270
1,bowled,268
2,leg before wicket,115
3,stumped,33
4,hit wicket,1


In [18]:
batting_dots = df.loc[
    df["ballfaced"].eq(1)
    & df["batruns"].eq(0)
].copy()

print("Batting dot-ball candidates:", len(batting_dots))

print("\nDismissal types among batting dot-ball candidates:")

display(
    batting_dots
    .loc[batting_dots["out"].eq(True)]
    .groupby("dismissal", dropna=False)
    .size()
    .rename("count")
    .reset_index()
)

print("\nBatting dot-ball candidates with nonzero team score:")

display(
    batting_dots.loc[
        batting_dots["score"].ne(0),
        [
            "p_match",
            "year",
            "inns",
            "over_num",
            "ball",
            "bat",
            "bowl",
            "batruns",
            "score",
            "byes",
            "legbyes",
            "out",
            "dismissal",
        ],
    ].head(20)
)

Batting dot-ball candidates: 11808

Dismissal types among batting dot-ball candidates:


,dismissal,count
0,bowled,268
1,caught,1270
2,hit wicket,1
3,leg before wicket,115
4,retired not out (hurt),2
5,retired out,1
6,run out,56
7,stumped,26



Batting dot-ball candidates with nonzero team score:


,p_match,year,inns,over_num,ball,bat,bowl,batruns,score,byes,legbyes,out,dismissal
1,1359475,2023,1,1,2,Devon Conway,Mohammed Shami,0,1,0,1,False,NaN
27,1359475,2023,1,5,4,Moeen Ali,Mohammed Shami,0,1,0,0,False,NaN
88,1359475,2023,1,15,3,Shivam Dube,Hardik Pandya,0,4,0,4,False,NaN
101,1359475,2023,1,17,4,Shivam Dube,Rashid Khan,0,1,1,0,False,NaN
117,1359475,2023,1,20,2,Mitchell Santner,Josh Little,0,1,0,1,False,NaN
130,1359475,2023,2,2,3,Wriddhiman Saha,Tushar Deshpande,0,1,0,1,False,NaN
141,1359475,2023,2,4,2,Wriddhiman Saha,Rajvardhan Hangargekar,0,1,0,0,False,NaN
146,1359475,2023,2,7,4,Shubman Gill,Ravindra Jadeja,0,1,0,1,False,NaN
237,1359475,2023,2,19,2,Rahul Tewatia,Deepak Chahar,0,4,0,4,False,NaN
303,1359476,2023,1,10,3,Bhanuka Rajapaksa,Varun Chakravarthy,0,1,0,1,False,NaN


In [19]:
print("Minimum over number:", df["over_num"].min())
print("Maximum over number:", df["over_num"].max())

print("\nDelivery records by over number:")

display(
    df.groupby("over_num")
    .size()
    .rename("delivery_records")
    .reset_index()
    .sort_values("over_num")
)

Minimum over number: 1
Maximum over number: 20

Delivery records by over number:


,over_num,delivery_records
0,1,1821
1,2,1838
2,3,1794
3,4,1802
4,5,1794
5,6,1774
6,7,1781
7,8,1779
8,9,1788
9,10,1776
